In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [4]:
import os
!pip install dagshub mlflow -q
from kaggle_secrets import UserSecretsClient
os.environ["DAGSHUB_USER_TOKEN"] = UserSecretsClient().get_secret("DAGSHUB_TOKEN")

import dagshub
dagshub.init(repo_owner='lkhiz23', repo_name='IEEE-CIS-Fraud-Detection', mlflow=True)

import mlflow
print("Connected ✓")

Accessing as lkhiz23

Initialized MLflow to track repo "lkhiz23/IEEE-CIS-Fraud-Detection"

Repository lkhiz23/IEEE-CIS-Fraud-Detection initialized!

Connected ✓


# Model Experiment - AdaBoost

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PATH = '/kaggle/input/competitions/ieee-fraud-detection/'

train_tx = pd.read_csv(PATH + 'train_transaction.csv')
train_id = pd.read_csv(PATH + 'train_identity.csv')
test_tx  = pd.read_csv(PATH + 'test_transaction.csv')
test_id  = pd.read_csv(PATH + 'test_identity.csv')

train = train_tx.merge(train_id, on='TransactionID', how='left')
test  = test_tx.merge(test_id,  on='TransactionID', how='left')

print("Train shape:", train.shape)
print("Test shape: ", test.shape)
train.head()

Train shape: (590540, 434)
Test shape:  (506691, 433)


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


In [6]:
from sklearn.model_selection import train_test_split

X = train.drop(columns=['isFraud', 'TransactionID'])
y = train['isFraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"Fraud rate in train: {y_train.mean():.4f}")
print(f"Fraud rate in test:  {y_test.mean():.4f}")

X_train: (472432, 432)
X_test:  (118108, 432)
Fraud rate in train: 0.0350
Fraud rate in test:  0.0350


## Data Cleaning

In [7]:
with mlflow.start_run(run_name="AdaBoost_Cleaning"):
    missing = X_train.isnull().mean()
    drop_cols = missing[missing > 0.8].index.tolist()
    X_train = X_train.drop(columns=drop_cols)
    X_test  = X_test.drop(columns=drop_cols)
    print(f"Dropped: {len(drop_cols)} | Remaining: {X_train.shape[1]}")
    mlflow.log_param("missing_threshold", 0.8)
    mlflow.log_param("dropped_cols", len(drop_cols))
    mlflow.log_param("remaining_features", X_train.shape[1])

Dropped: 74 | Remaining: 358
🏃 View run AdaBoost_Cleaning at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/1bea4356fec441c69b3710eed6ca55cd
🧪 View experiment at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0


## Feature Engineering

In [8]:
with mlflow.start_run(run_name="AdaBoost_Feature_Engineering"):
    def feature_engineering(df):
        df = df.copy()
        df['hour']  = (df['TransactionDT'] // 3600) % 24
        df['day']   = (df['TransactionDT'] // (3600 * 24)) % 7
        df['month'] = (df['TransactionDT'] // (3600 * 24 * 30)) % 12
        df['log_TransactionAmt'] = np.log1p(df['TransactionAmt'])
        df['cents'] = df['TransactionAmt'] - np.floor(df['TransactionAmt'])
        df['uid']  = df['card1'].astype(str) + '_' + df['card2'].astype(str)
        df['uid2'] = df['uid'] + '_' + df['card3'].astype(str)
        df['email_match']   = (df['P_emaildomain'] == df['R_emaildomain']).astype(int)
        df['P_email_count'] = df['P_emaildomain'].map(df['P_emaildomain'].value_counts())
        df['R_email_count'] = df['R_emaildomain'].map(df['R_emaildomain'].value_counts())
        df['card1_count'] = df['card1'].map(df['card1'].value_counts())
        df['uid_count']   = df['uid'].map(df['uid'].value_counts())
        if 'id_30' in df.columns:
            df['OS'] = df['id_30'].str.extract(r'^([a-zA-Z\s]+)')
        if 'id_31' in df.columns:
            df['browser'] = df['id_31'].str.extract(r'^([a-zA-Z\s]+)')
        return df
    X_train = feature_engineering(X_train)
    X_test  = feature_engineering(X_test)
    print(f"Shape after FE: {X_train.shape}")
    mlflow.log_param("total_features", X_train.shape[1])

Shape after FE: (472432, 371)
🏃 View run AdaBoost_Feature_Engineering at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/e83611fe3e024a3cadf472267062a774
🧪 View experiment at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0


## Encoding and Imputation

In [9]:
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer

with mlflow.start_run(run_name="AdaBoost_Encoding"):
    cat_cols = X_train.select_dtypes(include='object').columns.tolist()
    for col in cat_cols:
        le = LabelEncoder()
        combined = pd.concat([X_train[col], X_test[col]], axis=0).astype(str)
        le.fit(combined)
        X_train[col] = le.transform(X_train[col].astype(str))
        X_test[col]  = le.transform(X_test[col].astype(str))
    print(f"Label encoded {len(cat_cols)} columns")
    imputer = SimpleImputer(strategy='median')
    X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
    X_test  = pd.DataFrame(imputer.transform(X_test),      columns=X_test.columns)
    print(f"Imputed. Final shape: {X_train.shape}")
    mlflow.log_param("encoding", "LabelEncoder")
    mlflow.log_param("imputation", "median")
    mlflow.log_param("scaling", "None - not needed for trees")

Label encoded 29 columns
Imputed. Final shape: (472432, 371)
🏃 View run AdaBoost_Encoding at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/4f1f592548ca4b858626c8f3d8f8149d
🧪 View experiment at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0


## Feature Selection

In [11]:
from sklearn.feature_selection import SelectKBest, chi2, f_classif
from sklearn.preprocessing import MinMaxScaler

with mlflow.start_run(run_name="AdaBoost_Feature_Selection"):
    # - method 1: correlation filter -
    correlations = pd.Series(
        np.abs(np.corrcoef(X_train.T, y_train)[-1, :-1]),
        index=X_train.columns
    )
    low_corr_cols = correlations[correlations < 0.005].index.tolist()
    X_train_fs = X_train.drop(columns=low_corr_cols)
    X_test_fs  = X_test.drop(columns=low_corr_cols)
    print(f"Correlation filter removed: {len(low_corr_cols)} features")
    print(f"Remaining: {X_train_fs.shape[1]}")

    # - method 2: SelectKBest with f_classif - different from previous models -
    k = 80
    selector = SelectKBest(f_classif, k=k)
    X_train_fs = pd.DataFrame(
        selector.fit_transform(X_train_fs, y_train),
        columns=X_train_fs.columns[selector.get_support()]
    )
    X_test_fs = pd.DataFrame(
        selector.transform(X_test_fs),
        columns=X_train_fs.columns
    )
    print(f"SelectKBest kept: {k} features")
    print(f"Final shape: {X_train_fs.shape}")

    mlflow.log_param("corr_threshold", 0.005)
    mlflow.log_param("corr_removed", len(low_corr_cols))
    mlflow.log_param("selection_method", "SelectKBest_f_classif")
    mlflow.log_param("k", k)
    mlflow.log_metric("final_features", X_train_fs.shape[1])

Correlation filter removed: 73 features
Remaining: 298
SelectKBest kept: 80 features
Final shape: (472432, 80)
🏃 View run AdaBoost_Feature_Selection at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/05c50de56e9743f69dbfd0a13bc9f1c0
🧪 View experiment at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0


## Training

In [12]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_score

# - underfit - too few estimators -
with mlflow.start_run(run_name="AdaBoost_Underfit"):
    model = AdaBoostClassifier(n_estimators=10, learning_rate=0.1, random_state=42)
    model.fit(X_train_fs, y_train)
    train_auc = roc_auc_score(y_train, model.predict_proba(X_train_fs)[:,1])
    test_auc  = roc_auc_score(y_test,  model.predict_proba(X_test_fs)[:,1])
    print(f"Underfit | Train: {train_auc:.4f} | Test: {test_auc:.4f}")
    mlflow.log_params({"n_estimators": 10, "learning_rate": 0.1})
    mlflow.log_metrics({"train_auc": train_auc, "test_auc": test_auc})

# - overfit - too many estimators, high learning rate -
with mlflow.start_run(run_name="AdaBoost_Overfit"):
    model = AdaBoostClassifier(n_estimators=500, learning_rate=1.0, random_state=42)
    model.fit(X_train_fs, y_train)
    train_auc = roc_auc_score(y_train, model.predict_proba(X_train_fs)[:,1])
    test_auc  = roc_auc_score(y_test,  model.predict_proba(X_test_fs)[:,1])
    print(f"Overfit  | Train: {train_auc:.4f} | Test: {test_auc:.4f}")
    mlflow.log_params({"n_estimators": 500, "learning_rate": 1.0})
    mlflow.log_metrics({"train_auc": train_auc, "test_auc": test_auc})

# - tuned models -
for n_est in [50, 100, 200]:
    for lr in [0.01, 0.1, 0.5]:
        with mlflow.start_run(run_name=f"AdaBoost_n{n_est}_lr{lr}"):
            model = AdaBoostClassifier(
                n_estimators=n_est,
                learning_rate=lr,
                random_state=42
            )
            cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
            cv_scores = cross_val_score(model, X_train_fs, y_train, cv=cv, scoring='roc_auc')
            model.fit(X_train_fs, y_train)
            
            y_pred       = model.predict(X_test_fs)
            y_pred_proba = model.predict_proba(X_test_fs)[:,1]
            train_auc    = roc_auc_score(y_train, model.predict_proba(X_train_fs)[:,1])
            test_auc     = roc_auc_score(y_test, y_pred_proba)
            f1           = f1_score(y_test, y_pred)
            precision    = precision_score(y_test, y_pred)
            recall       = recall_score(y_test, y_pred)
            
            print(f"n={n_est} lr={lr} | Train: {train_auc:.4f} | Test: {test_auc:.4f} | F1: {f1:.4f} | CV: {cv_scores.mean():.4f}")
            mlflow.log_params({"n_estimators": n_est, "learning_rate": lr})
            mlflow.log_metrics({
                "train_auc": train_auc,
                "test_auc": test_auc,
                "cv_auc_mean": cv_scores.mean(),
                "cv_auc_std": cv_scores.std(),
                "test_f1": f1,
                "test_precision": precision,
                "test_recall": recall
            })

Underfit | Train: 0.6405 | Test: 0.6418
🏃 View run AdaBoost_Underfit at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/651f99615b95441a85f6c4ba58f655b9
🧪 View experiment at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0
Overfit  | Train: 0.7410 | Test: 0.7474
🏃 View run AdaBoost_Overfit at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/3474a82bfaee41c5b722fb452a50a7e4
🧪 View experiment at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


n=50 lr=0.01 | Train: 0.6447 | Test: 0.6455 | F1: 0.0000 | CV: 0.6447
🏃 View run AdaBoost_n50_lr0.01 at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/4ef7be656dae489ca385d4d5a63d5c8d
🧪 View experiment at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0
n=50 lr=0.1 | Train: 0.7197 | Test: 0.7262 | F1: 0.1279 | CV: 0.7222
🏃 View run AdaBoost_n50_lr0.1 at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/6ac02622b6f949cb913b1602962e95d4
🧪 View experiment at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0
n=50 lr=0.5 | Train: 0.7175 | Test: 0.7230 | F1: 0.2724 | CV: 0.7156
🏃 View run AdaBoost_n50_lr0.5 at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/88efe375cb2d409688af8bebeb897c72
🧪 View experiment at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


n=100 lr=0.01 | Train: 0.6446 | Test: 0.6454 | F1: 0.0000 | CV: 0.6446
🏃 View run AdaBoost_n100_lr0.01 at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/457b742150bb45bdb11f90dec25b9562
🧪 View experiment at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0
n=100 lr=0.1 | Train: 0.7237 | Test: 0.7299 | F1: 0.1446 | CV: 0.7238
🏃 View run AdaBoost_n100_lr0.1 at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/28b08dfaf54d45968af2bf4eb8a64245
🧪 View experiment at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0
n=100 lr=0.5 | Train: 0.7201 | Test: 0.7262 | F1: 0.2727 | CV: 0.7221
🏃 View run AdaBoost_n100_lr0.5 at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/401ae9dd2f8c4b7da2699980922e985e
🧪 View experiment at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


n=200 lr=0.01 | Train: 0.7131 | Test: 0.7174 | F1: 0.0000 | CV: 0.7134
🏃 View run AdaBoost_n200_lr0.01 at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/bc8fc54954414fd687f5a355f704fa77
🧪 View experiment at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0
n=200 lr=0.1 | Train: 0.7241 | Test: 0.7303 | F1: 0.1556 | CV: 0.7254
🏃 View run AdaBoost_n200_lr0.1 at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/b8095df6bf8d48b68740687d717a48fb
🧪 View experiment at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0
n=200 lr=0.5 | Train: 0.7267 | Test: 0.7325 | F1: 0.2719 | CV: 0.7245
🏃 View run AdaBoost_n200_lr0.5 at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/37bbb57215224abba36728768a4f1808
🧪 View experiment at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0


In [13]:
with mlflow.start_run(run_name="AdaBoost_Best_Pipeline"):
    from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, classification_report
    
    best_model = AdaBoostClassifier(
        n_estimators=200,
        learning_rate=0.5,
        random_state=42
    )
    best_model.fit(X_train_fs, y_train)
    
    y_pred       = best_model.predict(X_test_fs)
    y_pred_proba = best_model.predict_proba(X_test_fs)[:,1]
    train_auc    = roc_auc_score(y_train, best_model.predict_proba(X_train_fs)[:,1])
    test_auc     = roc_auc_score(y_test, y_pred_proba)
    f1           = f1_score(y_test, y_pred)
    precision    = precision_score(y_test, y_pred)
    recall       = recall_score(y_test, y_pred)
    
    print(f"AUC: {test_auc:.4f} | F1: {f1:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f}")
    print(classification_report(y_test, y_pred))
    
    mlflow.log_params({"n_estimators": 200, "learning_rate": 0.5})
    mlflow.log_metrics({
        "train_auc": train_auc,
        "test_auc": test_auc,
        "test_f1": f1,
        "test_precision": precision,
        "test_recall": recall
    })
    mlflow.sklearn.log_model(
        best_model,
        "adaboost_model",
        registered_model_name="AdaBoost_Fraud"
    )
    print("Model saved.")

AUC: 0.7325 | F1: 0.2719 | Precision: 0.6935 | Recall: 0.1691
              precision    recall  f1-score   support

           0       0.97      1.00      0.98    113975
           1       0.69      0.17      0.27      4133

    accuracy                           0.97    118108
   macro avg       0.83      0.58      0.63    118108
weighted avg       0.96      0.97      0.96    118108



2026/05/03 12:27:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 12:28:00 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'AdaBoost_Fraud'.
2026/05/03 12:28:18 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: AdaBoost_Fraud, version 1
Created version '1' of model 'AdaBoost_Fraud'.


Model saved.
🏃 View run AdaBoost_Best_Pipeline at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/43ba53673a2f4166ac3888066bd668bf
🧪 View experiment at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0
